#  Ames Housing Price Prediction
### Linear Regression | KNN Regression | Support Vector Regression (SVR)

**Pipeline:** Dataset Overview -> EDA -> Data Analysis & Insights -> Preprocessing -> Pipelines -> Model Training -> Hyperparameter Tuning -> Evaluation & Comparison -> Feature Importance -> Final Model Selection

In [ ]:
# Setup & Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pathlib

from sklearn.ensemble import GradientBoostingRegressor

warnings.filterwarnings('ignore')
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler, RobustScaler,
    OneHotEncoder, OrdinalEncoder,
    FunctionTransformer
)
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error, mean_squared_error


SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.max_columns', 100)



In [ ]:
datapath = pathlib.Path("../../datasets/ames/ames_train.csv")
df = pd.read_csv(datapath)

# In Ames housing dataset, 'NA' in specific columns represents the absence of a feature
# (e.g., 'No Basement', 'No Garage', 'No Pool'), NOT missing data.
# We replace NaN with 'None' string so it acts as a valid, informative category.
NaNoneCols = [
    'Alley', 'MasVnrType', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
    'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu', 'GarageType',
    'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature'
]

for col in NaNoneCols:
    if col in df.columns:
        df[col] = df[col].fillna('None')

display(df.head())

In [ ]:
# Check for duplicate rows (excluding the arbitrary 'Id' column)
n_dup = df.drop(columns='Id').duplicated().sum()
print(f"Number of duplicate rows (excluding Id): {n_dup}")
# Identify remaining genuine missing values
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
# Build a clean summary table safely using pd.concat
missing_df = pd.concat([missing.rename('Missing Count'), missing_pct.rename('Missing %')], axis=1)
print("\nGenuine Missing Values Summary:")
display(missing_df)

In [ ]:
# Classify Features into Numerical, Nominal, and Ordinal Types
# 1. Ordinal Features: Categorical variables with a clear inherent order
ORDINAL_COLS = [
    'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
    'BsmtFinType1', 'BsmtFinType2', 'HeatingQC', 'KitchenQual',
    'FireplaceQu', 'GarageFinish', 'GarageQual', 'GarageCond',
    'PoolQC', 'Fence', 'Functional'
]

# 2. Numerical Features: Continuous and discrete measurements (excluding Id and SalePrice target)
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols = [c for c in num_cols if c not in ['Id', 'SalePrice']]

# 3. Nominal Features: Categorical variables without any natural ordering
all_cat  = df.select_dtypes(include='object').columns.tolist()
nom_cols = [c for c in all_cat if c not in ORDINAL_COLS]
print(f"Total Numerical Features : {len(num_cols)}")
print(f"Total Nominal Features   : {len(nom_cols)}")
print(f"Total Ordinal Features   : {len(ORDINAL_COLS)}")
print(f"Total Predictors Count   : {len(num_cols) + len(nom_cols) + len(ORDINAL_COLS)}")

In [ ]:
# SalePrice Distribution Analysis & Right-Skewness Mitigation

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
# 1. Raw SalePrice Distribution
axes[0].hist(df['SalePrice'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title(f"Raw SalePrice (Skewness: {df['SalePrice'].skew():.2f})", fontweight='bold')
axes[0].set_xlabel("SalePrice ($)")
axes[0].set_ylabel("Frequency")
# 2. Log-Transformed SalePrice: log(1 + SalePrice)
log_target = np.log1p(df['SalePrice'])
axes[1].hist(log_target, bins=50, color='darkorange', edgecolor='white')
axes[1].set_title(f"log(1 + SalePrice) (Skewness: {log_target.skew():.2f})", fontweight='bold')
axes[1].set_xlabel("log(1 + SalePrice)")
axes[1].set_ylabel("Frequency")
plt.tight_layout()
plt.show()
print(f"Skewness before log-transform : {df['SalePrice'].skew():.3f} (Significant positive skew)")
print(f"Skewness after log-transform  : {log_target.skew():.3f} (Nearly normal distribution)")

In [ ]:
# Inspect Skewness in Predictor Numerical Features
# Calculate absolute skewness for all numerical features
skew_series = df[num_cols].skew().sort_values(ascending=False)
highly_skewed = skew_series[skew_series.abs() > 0.75]
plt.figure(figsize=(12, 4))
colors = ['crimson' if abs(v) > 1.5 else 'steelblue' for v in highly_skewed.values]
plt.bar(highly_skewed.index, highly_skewed.values, color=colors)
plt.axhline(0.75,  color='green', linestyle='--', label='Threshold |skew| = 0.75')
plt.axhline(-0.75, color='green', linestyle='--')
plt.xticks(rotation=90)
plt.title(f"Numerical Features with |skew| > 0.75 ({len(highly_skewed)} features)", fontweight='bold')
plt.ylabel("Skewness")
plt.legend()
plt.tight_layout()
plt.show()
print(f"{len(highly_skewed)} numerical features have high skewness and will be log-transformed in the pipeline.")



In [ ]:
corr_matrix = df[num_cols + ['SalePrice']].corr()
top15 = corr_matrix['SalePrice'].abs().sort_values(ascending=False).head(15).index
plt.figure(figsize=(11, 8))
sns.heatmap(corr_matrix.loc[top15, top15], annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title("Correlation Heatmap: Top 15 Features vs SalePrice", fontweight='bold')
plt.tight_layout()
plt.show()
# 2. Scatter plot to identify the two confirmed extreme outliers
plt.figure(figsize=(8, 4))
plt.scatter(df['GrLivArea'], df['SalePrice'], alpha=0.4, color='steelblue', label='Standard Homes')
# Filter the two famous extreme outliers (huge square footage but sold unusually cheap)
outliers = df[(df['GrLivArea'] > 4000) & (df['SalePrice'] < 300000)]
plt.scatter(outliers['GrLivArea'], outliers['SalePrice'], color='red', s=70, label='Extreme Outliers (To Drop)', zorder=5)
plt.xlabel('GrLivArea (Above Grade Living Area sq ft)')
plt.ylabel('SalePrice ($)')
plt.title('GrLivArea vs SalePrice - Outlier Identification', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#  Answering Key Domain & Analytical Questions with Real Values
# Q1: Overall summary statistics for SalePrice
print("SalePrice Summary")
print(f"Mean   : ${df['SalePrice'].mean():,.0f}")
print(f"Median : ${df['SalePrice'].median():,.0f}")
print(f"Min    : ${df['SalePrice'].min():,.0f}")
print(f"Max    : ${df['SalePrice'].max():,.0f}")
# Q2: Top 3 & Bottom 3 Neighborhoods by Average Price
neigh_avg = df.groupby('Neighborhood')['SalePrice'].mean().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
neigh_avg.head(10).plot(kind='bar', ax=axes[0], color='steelblue', title='Top 10 Expensive Neighborhoods')
axes[0].set_ylabel('Avg Sale price (USD)')
neigh_avg.tail(10).plot(kind='bar', ax=axes[1], color='crimson', title='Bottom 10 Cheap Neighborhoods')
axes[1].set_ylabel('Avg Sale price (USD)')
plt.tight_layout()
plt.show()
print("\nTop 3 Neighborhoods:")
for n, p in neigh_avg.head(3).items():
    print(f"  {n:<12}: ${p:,.0f}")
print("\nBottom 3 Neighborhoods:")
for n, p in neigh_avg.tail(3).items():
    print(f"  {n:<12}: ${p:,.0f}")

has_bsmt = (df['TotalBsmtSF'] > 0).map({True: 'Has Basement', False: 'No Basement'})
bsmt_avg = df.groupby(has_bsmt)['SalePrice'].mean()
diff = bsmt_avg['Has Basement'] - bsmt_avg['No Basement']
print(f"\nAverage Price with Basement    : ${bsmt_avg['Has Basement']:,.0f}")
print(f"Average Price without Basement : ${bsmt_avg['No Basement']:,.0f}")
print(f"Conclusion: Having a basement adds an average premium of ${diff:,.0f}.")

In [ ]:
# Data Cleaning — Drop High Missing Columns, Id & Outliers
# 1. Drop columns with >80% missingness (Alley, PoolQC, MiscFeature) + unneeded Id
miss_pct = df.isnull().mean()
high_miss = miss_pct[miss_pct > 0.80].index.tolist()
print(f"Columns dropped due to >80% missing values: {high_miss}")
df = df.drop(columns=high_miss + ['Id'])

before_count = len(df)
df = df[~((df['GrLivArea'] > 4000) & (df['SalePrice'] < 300000))]
print(f"Extreme outlier rows removed: {before_count - len(df)}")
print(f"Cleaned dataset shape: {df.shape}")

In [ ]:
# Feature Engineering
def transformXF(df_):
    df_ = df_.copy()

    df_['TotalSF'] = df_['TotalBsmtSF'] + df_['1stFlrSF'] + df_['2ndFlrSF']

    df_['TotalBath'] = (df_['FullBath'] + 0.5 * df_['HalfBath']
                        + df_['BsmtFullBath'] + 0.5 * df_['BsmtHalfBath'])

    df_['HouseAge'] = df_['YrSold'] - df_['YearBuilt']
    df_['RemodAge'] = df_['YrSold'] - df_['YearRemodAdd']
    # 4. Total exterior porch area
    df_['TotalPorchSF'] = (df_['OpenPorchSF'] + df_['EnclosedPorch']
                           + df_['3SsnPorch'] + df_['ScreenPorch'] + df_['WoodDeckSF'])

    df_['HasPool']      = (df_['PoolArea'] > 0).astype(int)
    df_['HasGarage']    = (df_['GarageArea'] > 0).astype(int)
    df_['HasBasement']  = (df_['TotalBsmtSF'] > 0).astype(int)
    df_['WasRemodeled'] = (df_['YearRemodAdd'] != df_['YearBuilt']).astype(int)

    df_['QualArea'] = df_['OverallQual'] * df_['GrLivArea']

    df_.drop(columns=['GarageYrBlt', 'GarageCars'])

    return df_

y_full = np.log1p(df['SalePrice'])
X_full = transformXF(df.drop(columns='SalePrice'))

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_full, y_full, test_size=0.15, random_state=SEED
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=round(0.15 / 0.85, 6), random_state=SEED
)
print(f"Training set   : {X_train.shape[0]} rows ({X_train.shape[0] / len(X_full):.0%})")
print(f"Validation set : {X_val.shape[0]} rows ({X_val.shape[0] / len(X_full):.0%})")
print(f"Test set       : {X_test.shape[0]} rows ({X_test.shape[0] / len(X_full):.0%})")

In [ ]:
remaining_cols     = X_train.columns.tolist()
ORDINAL_COLS_FINAL = [c for c in ORDINAL_COLS if c in remaining_cols]
num_features       = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
nom_features       = [c for c in X_train.select_dtypes(include='object').columns if c not in ORDINAL_COLS_FINAL]

train_skew       = X_train[num_features].skew().abs()
skewed_feats     = train_skew[train_skew > 0.75].index.tolist()
non_skewed_feats = [c for c in num_features if c not in skewed_feats]

QUALITY_ORDER = ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
ordinal_categories = {
    'ExterQual'   : QUALITY_ORDER,
    'ExterCond'   : QUALITY_ORDER,
    'BsmtQual'    : QUALITY_ORDER,
    'BsmtCond'    : QUALITY_ORDER,
    'BsmtExposure': ['None', 'No', 'Mn', 'Av', 'Gd'],
    'BsmtFinType1': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'BsmtFinType2': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'HeatingQC'   : QUALITY_ORDER,
    'KitchenQual' : QUALITY_ORDER,
    'FireplaceQu' : QUALITY_ORDER,
    'GarageFinish': ['None', 'Unf', 'RFn', 'Fin'],
    'GarageQual'  : QUALITY_ORDER,
    'GarageCond'  : QUALITY_ORDER,
    'PoolQC'      : QUALITY_ORDER,
    'Fence'       : ['None', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv'],
    'Functional'  : ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
}

ord_cats_list = [ordinal_categories[c] for c in ORDINAL_COLS_FINAL if c in ordinal_categories]

pipe_num_skewed = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log1p',   FunctionTransformer(np.log1p, validate=False)),
    ('scaler',  RobustScaler())
])

pipe_num_normal = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

pipe_nominal = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
])

pipe_ordinal = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('oe',      OrdinalEncoder(categories=ord_cats_list, handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer([
    ('num_skewed', pipe_num_skewed, skewed_feats),
    ('num_normal', pipe_num_normal, non_skewed_feats),
    ('nominal',    pipe_nominal,    nom_features),
    ('ordinal',    pipe_ordinal,    ORDINAL_COLS_FINAL),
], remainder='drop')

print("Preprocessing ColumnTransformer built successfully!")
print(f"  - Skewed numerical features     : {len(skewed_feats)}")
print(f"  - Non-skewed numerical features : {len(non_skewed_feats)}")
print(f"  - Nominal categorical features  : {len(nom_features)}")
print(f"  - Ordinal categorical features  : {len(ORDINAL_COLS_FINAL)}")

In [ ]:
Lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

Lr.fit(X_train, y_train)


svr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', SVR(C=0.76, epsilon=0.09))
])

# Parameter grid for SVR as suggested in guidelines
svrParamGrid = {
    'regressor__C': [0.01 ,0.1, 1, 10],
    'regressor__kernel': ['rbf', 'linear'],
    'regressor__gamma': ['auto', 0.1, 1, 10],
    'regressor__degree': [2,3],
}

svrGrid = GridSearchCV(
    estimator=svr,
    param_grid=svrParamGrid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)
svrGrid.fit(X_train, y_train)


gbr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(loss='huber', learning_rate=0.04, n_estimators=174, max_depth=3))
])

gbr.fit(X_train, y_train)



In [ ]:
models = {
    'Linear Regression': Lr,
    'SVR': svrGrid.best_estimator_,
    'Gradient Boosting': gbr
}

evaluation_results = []

for name, model in models.items():
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Metrics on Test Set
    mae = mean_absolute_error(y_test, y_test_pred)
    mse = mean_squared_error(y_test, y_test_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_test_pred)
    train_r2 = r2_score(y_train, y_train_pred)

    evaluation_results.append({
        'Model': name,
        'Train R2': train_r2,
        'Test R2': r2,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse
    })

results_df = pd.DataFrame(evaluation_results).set_index('Model')
display(results_df)

